# 🔁 Data Augmentation
**One-line description:** Expand your training data synthetically to improve model robustness when real data is scarce or imbalanced.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/05_data_augmentation.ipynb)


In [ ]:
# Install required libraries (run this cell first in Google Colab)
!pip install imbalanced-learn scikit-learn pandas numpy matplotlib seaborn faker --quiet


## 📖 What is Data Augmentation?

Data augmentation creates **new, synthetic training examples** from existing ones to increase dataset size and diversity without collecting more real data.

**Analogy:** Imagine you're a musician learning a new piece. Instead of just listening to the original recording, you also hear it played in different keys (transposed), at different tempos, and with slight variations. Each variation helps you understand the melody more deeply. That's augmentation — same core information, different perspectives.

### When to Use Augmentation:
| Scenario | Augmentation Strategy |
|----------|--------------------|
| Small dataset | Noise injection, bootstrapping |
| Imbalanced classes | SMOTE, ADASYN |
| Privacy constraints | Synthetic data generation |
| Image data | Flip, rotate, crop, color jitter |
| Text data | Synonym replacement, back-translation |


## 💡 Why Does It Matter?

- Models trained on small datasets tend to **overfit**
- Class-imbalanced data causes models to **ignore minority classes**
- Augmentation can be the difference between a 70% and 90% accurate model on rare events
- Synthetic data allows experimentation without risking sensitive real data


## ⚙️ How Does It Work?

We'll cover 5 tabular augmentation techniques:
1. **Noise Injection** — add small Gaussian noise to numeric features
2. **Bootstrapping** — resample with replacement to create new datasets
3. **SMOTE** — Synthetic Minority Over-sampling TEchnique (interpolates between minority samples)
4. **ADASYN** — Adaptive Synthetic Sampling (focuses on harder-to-classify regions)
5. **Faker-based Synthetic Generation** — create entirely new realistic records


## 🛠️ Hands-on Code

### Step 1: Create an Imbalanced Binary Classification Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline
from faker import Faker
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
fake = Faker()
Faker.seed(42)

# Create imbalanced dataset (90% negative, 10% positive — like fraud detection)
X, y = make_classification(
    n_samples=2000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    weights=[0.90, 0.10],  # 90/10 imbalance
    random_state=42,
    flip_y=0.01
)

feature_names = ['TransactionAmt', 'AccountAge', 'NumTransactions',
                 'AvgTransAmt', 'DaysLastLogin', 'DeviceRiskScore',
                 'GeoRiskScore', 'TimeOfDay']

df = pd.DataFrame(X, columns=feature_names)
df['Label'] = y

print("Dataset shape:", df.shape)
print(f"Class distribution:")
print(df['Label'].value_counts())
print(f"\nMinority class: {df['Label'].sum()} samples ({df['Label'].mean()*100:.1f}%)")


### Step 2: Noise Injection Augmentation


In [ ]:
# --- Noise Injection: add small Gaussian noise to create new samples ---
def noise_augmentation(X_minority, n_augmented, noise_std=0.05):
    # Add Gaussian noise to minority class samples to create augmented samples
    # noise_std: fraction of feature std to use as noise magnitude
    augmented_samples = []
    feature_stds = X_minority.std(axis=0)

    for _ in range(n_augmented):
        # Pick a random minority sample
        idx = np.random.randint(0, len(X_minority))
        sample = X_minority[idx].copy()

        # Add small Gaussian noise proportional to feature std
        noise = np.random.normal(0, noise_std * feature_stds)
        augmented_sample = sample + noise
        augmented_samples.append(augmented_sample)

    return np.array(augmented_samples)

# Get minority class samples
X_minority = df[df['Label'] == 1][feature_names].values
n_to_augment = len(df[df['Label'] == 0]) - len(X_minority)  # How many to create

X_noise_aug = noise_augmentation(X_minority, n_to_augment, noise_std=0.05)
y_noise_aug = np.ones(n_to_augment)

X_balanced_noise = np.vstack([df[feature_names].values, X_noise_aug])
y_balanced_noise = np.concatenate([df['Label'].values, y_noise_aug])

print(f'Minority class before noise augmentation: {len(X_minority)}')
print(f'Samples created by noise injection:       {n_to_augment}')
print(f'New class distribution: {pd.Series(y_balanced_noise).value_counts().to_dict()}')


### Step 3: SMOTE Augmentation


In [ ]:
# --- SMOTE: Synthetic Minority Over-sampling TEchnique ---
# SMOTE creates synthetic samples by interpolating between existing minority samples
# It selects a minority sample and one of its k nearest neighbors, then creates
# a new sample along the line segment between them

smote = SMOTE(sampling_strategy='auto',  # balance to 1:1 ratio
              k_neighbors=5,
              random_state=42)

X_smote, y_smote = smote.fit_resample(df[feature_names].values, df['Label'].values)

print("SMOTE Augmentation:")
print(f"  Before: {pd.Series(df['Label'].values).value_counts().to_dict()}")
print(f"  After:  {pd.Series(y_smote).value_counts().to_dict()}")
print(f"  New synthetic samples created: {len(X_smote) - len(df)}")

# --- ADASYN: Adaptive Synthetic Sampling ---
# ADASYN generates more synthetic data for harder-to-classify minority samples
adasyn = ADASYN(sampling_strategy='auto',
                n_neighbors=5,
                random_state=42)

X_adasyn, y_adasyn = adasyn.fit_resample(df[feature_names].values, df['Label'].values)

print("\nADASYN Augmentation:")
print(f"  Before: {pd.Series(df['Label'].values).value_counts().to_dict()}")
print(f"  After:  {pd.Series(y_adasyn).value_counts().to_dict()}")
print(f"  ADASYN adapts density of synthetic samples to difficulty of classification")


In [ ]:
# --- Visualization 1: Before vs After augmentation ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_configs = [
    (df[feature_names].values, df['Label'].values, 'Original (Imbalanced)'),
    (X_smote, y_smote, 'After SMOTE'),
    (X_adasyn, y_adasyn, 'After ADASYN'),
]

for ax, (X_plot, y_plot, title) in zip(axes, plot_configs):
    mask0 = y_plot == 0
    mask1 = y_plot == 1
    ax.scatter(X_plot[mask0, 0], X_plot[mask0, 1], alpha=0.3, s=8,
               color='steelblue', label=f'Class 0 (n={mask0.sum()})')
    ax.scatter(X_plot[mask1, 0], X_plot[mask1, 1], alpha=0.5, s=8,
               color='coral', label=f'Class 1 (n={mask1.sum()})')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1 (TransactionAmt)')
    ax.set_ylabel('Feature 2 (AccountAge)')
    ax.legend(fontsize=8)

plt.suptitle('Data Augmentation Techniques: Before and After', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('augmentation_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 4: Bootstrapping


In [ ]:
# --- Bootstrapping: sample with replacement to create new datasets ---
# Useful for: estimating model variance, ensemble methods (Random Forest uses this!)

def bootstrap_sample(X, y, n_samples=None):
    # Create a bootstrap sample (sample with replacement)
    n = len(X) if n_samples is None else n_samples
    indices = np.random.choice(len(X), size=n, replace=True)  # with replacement!
    return X[indices], y[indices]

X_vals = df[feature_names].values
y_vals = df['Label'].values

# Create 5 bootstrap samples (used in bagging/Random Forest)
bootstrap_class_ratios = []
for i in range(5):
    X_boot, y_boot = bootstrap_sample(X_vals, y_vals)
    ratio = y_boot.mean()
    bootstrap_class_ratios.append(ratio)
    print(f"Bootstrap sample {i+1}: {len(X_boot)} samples, class 1 ratio: {ratio:.3f}")

print(f"\nOriginal class 1 ratio: {y_vals.mean():.3f}")
print(f"Mean bootstrap ratio:   {np.mean(bootstrap_class_ratios):.3f}")
print("\nNote: bootstrapping preserves the original class ratio on average")


### Step 5: Faker-based Synthetic Data Generation


In [ ]:
# --- Synthetic data generation using Faker ---
# Useful for: creating realistic datasets for testing, privacy-preserving data sharing

def generate_synthetic_record(fake_gen, label=0):
    # Generate a single synthetic record with realistic values
    if label == 0:  # Normal transaction
        return {
            'Name': fake_gen.name(),
            'Email': fake_gen.email(),
            'TransactionAmt': round(np.random.exponential(50), 2),
            'AccountAge': np.random.randint(30, 3650),  # days
            'NumTransactions': np.random.randint(1, 200),
            'AvgTransAmt': round(np.random.normal(75, 30), 2),
            'Country': fake_gen.country_code(),
            'Label': 0
        }
    else:  # Fraudulent transaction
        return {
            'Name': fake_gen.name(),
            'Email': fake_gen.email(),
            'TransactionAmt': round(np.random.exponential(500), 2),  # larger amounts
            'AccountAge': np.random.randint(1, 60),                  # new accounts
            'NumTransactions': np.random.randint(1, 5),               # few transactions
            'AvgTransAmt': round(np.random.normal(400, 100), 2),
            'Country': fake_gen.country_code(),
            'Label': 1
        }

# Generate synthetic dataset with realistic imbalance
synthetic_records = (
    [generate_synthetic_record(fake, label=0) for _ in range(950)] +
    [generate_synthetic_record(fake, label=1) for _ in range(50)]
)
np.random.shuffle(synthetic_records)
df_synthetic = pd.DataFrame(synthetic_records)

print("Synthetic dataset generated with Faker:")
print(df_synthetic.shape)
print("\nSample records:")
print(df_synthetic.head(3).to_string())
print(f"\nClass distribution: {df_synthetic['Label'].value_counts().to_dict()}")


In [ ]:
# --- Visualization 2: Distribution comparison — Original vs Augmented (SMOTE) ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, feat in enumerate(feature_names):
    ax = axes.flat[i]
    # Original minority
    orig_min = df[df['Label']==1][feat].values
    # SMOTE augmented minority
    smote_df = pd.DataFrame(X_smote, columns=feature_names)
    smote_df['Label'] = y_smote
    smote_min = smote_df[smote_df['Label']==1][feat].values

    ax.hist(orig_min, bins=20, alpha=0.6, color='coral', label='Original Minority', density=True)
    ax.hist(smote_min, bins=20, alpha=0.4, color='green', label='SMOTE Synthetic', density=True)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions: Original Minority vs SMOTE Synthetic Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('smote_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


## 📌 Key Takeaways

- **Only augment training data** — never touch validation or test sets
- **Noise injection** is simple but may move samples outside realistic bounds
- **SMOTE** creates interpolated samples — works well for most cases
- **ADASYN** focuses synthetic generation on harder boundary regions — can be more effective than SMOTE
- **Bootstrapping** is the foundation of ensemble methods (Random Forest, Bagging)
- **Faker** enables fully synthetic realistic datasets for testing and privacy
- Augmentation is not a substitute for better data collection when possible
- Always evaluate model with original (non-augmented) test set to measure true generalization
